In [ ]:
import sys

sys.path.append("/Users/ng27753/Astronomy_Research/hubersed/bin/prospector/")

In [ ]:
from save_results import load_galaxy_results
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import medfilt
import parameter_file as P

from fit_config import build_continuum_model, build_full_cue_model
import corner
from hubersed.prospector.utils import load_lines
from hubersed.conversion import flambda_to_maggies, ivar_flambda_to_ivar_maggies
from hubersed.style import *


EM_LINES_A = load_lines()["emission"]["wave_vac"]

In [ ]:
from hubersed.prospector.lsf import build_desi_resolution_matrix

wave = P.WAVE_OBS

R_MATRIX = build_desi_resolution_matrix(wave)

In [ ]:
# Load results
target_galaxy_id = "39628340549649241"
results = load_galaxy_results(target_galaxy_id, "../results/v2/cue/")
redshift = results["redshift"]
outlier_idx = results["outlier_idx"]

theta_best_cont = results["theta_map_cont"]
theta_best_full = results["theta_best_full"]

# flat_samples_cont = results["flat_samples_cont"]
flat_samples_full = results["flat_samples_full"]

# spec_cont = results["spec_cont"]

print(f"Loading results for galaxy {outlier_idx} at z={redshift:.3f}")

In [ ]:
# Rebuild obs
spec, unc, _, _, gal_id = P.get_outlier_info(outlier_idx)
wave_A = P.WAVE_OBS
spec_maggies = flambda_to_maggies(wave_A, spec)
ivar_maggies = ivar_flambda_to_ivar_maggies(wave_A, unc)
sigma_maggies = 1 / np.sqrt(np.where(ivar_maggies > 0, ivar_maggies, np.inf))

sps = P.build_sps()
fsps_waves = sps.ssp.emline_wavelengths
fsps_optical = fsps_waves[(fsps_waves > 3600) & (fsps_waves < 9824)]


mask = (sigma_maggies > 0) & np.isfinite(sigma_maggies)
mask_em = P.mask_spectral_lines(
    wave_A, mask, redshift, line_waves=fsps_optical, halfwidth_kms=1500.0
)

obs = P.build_obs(spec=spec_maggies, unc=sigma_maggies, mask=mask_em)
obs_full = P.build_obs(spec=spec_maggies, unc=sigma_maggies, mask=mask)

model, template = build_continuum_model(redshift)
model.set_parameters(theta_best_cont)
full_model, full_template = build_full_cue_model(
    continuum_template=template,
    theta_best_cont=theta_best_cont,
    cont_model=model,
    redshift=redshift,
)
full_model.set_parameters(theta_best_full)

pred_spec_cont, _ = model.predict(theta_best_cont, observations=obs, sps=sps)
spec_cont = pred_spec_cont[0]

In [ ]:
# ── Plot 1: Continuum fit ────────────────────────────────────────────────────
fig, axes = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
)
axes[0].step(
    wave[mask_em],
    medfilt(spec_maggies[mask_em], 3),
    where="mid",
    lw=0.7,
    c="k",
    label="Data",
)
axes[0].step(
    wave[mask_em],
    spec_cont[mask_em],
    where="mid",
    lw=1.2,
    c="tomato",
    label="Continuum MAP",
)
axes[0].fill_between(
    wave[mask_em],
    spec_maggies[mask_em] - sigma_maggies[mask_em],
    spec_maggies[mask_em] + sigma_maggies[mask_em],
    alpha=0.2,
    color="k",
    step="mid",
)
axes[0].set_ylabel("Flux [Maggies]")
axes[0].legend(frameon=False)

resid_cont = (spec_maggies[mask_em] - spec_cont[mask_em]) / sigma_maggies[mask_em]
axes[1].step(wave[mask_em], resid_cont, where="mid", lw=0.6, c="grey")
axes[1].axhline(0, c="k", lw=1.0, ls="--")
axes[1].axhline(+3, c="red", lw=0.6, ls=":", alpha=0.5)
axes[1].axhline(-3, c="red", lw=0.6, ls=":", alpha=0.5)
axes[1].set_ylabel(r"$(d-m)/\sigma$")
axes[1].set_xlabel("Wavelength [Å]")
axes[1].set_ylim(-5, 5)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Full nebular fit ─────────────────────────────────────────────────
if results.get("full_status") == "success":
    spec_full = results["spec_full"]

    fig, axes = plt.subplots(
        2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
    )
    axes[0].step(
        wave[mask_em],
        medfilt(spec_maggies[mask_em], 3),
        where="mid",
        lw=0.7,
        c="k",
        label="Data",
    )
    axes[0].step(
        wave[mask_em],
        spec_cont[mask_em],
        where="mid",
        lw=1.0,
        c="tomato",
        ls="--",
        label="Continuum only",
    )
    axes[0].step(
        wave[mask_em],
        spec_full[mask_em],
        where="mid",
        lw=1.2,
        c="C0",
        label="Full (with nebular)",
    )
    axes[0].fill_between(
        wave[mask_em],
        spec_maggies[mask_em] - sigma_maggies[mask_em],
        spec_maggies[mask_em] + sigma_maggies[mask_em],
        alpha=0.2,
        color="k",
        step="mid",
    )
    axes[0].set_ylabel("Flux [Maggies]")
    axes[0].legend(frameon=False)

    resid_full = (spec_maggies[mask_em] - spec_full[mask_em]) / sigma_maggies[mask_em]
    axes[1].step(wave[mask_em], resid_full, where="mid", lw=0.6, c="grey")
    axes[1].axhline(0, c="k", lw=1.0, ls="--")
    axes[1].axhline(+3, c="red", lw=0.6, ls=":", alpha=0.5)
    axes[1].axhline(-3, c="red", lw=0.6, ls=":", alpha=0.5)
    axes[1].set_ylabel(r"$(d-m)/\sigma$")
    axes[1].set_xlabel("Wavelength [Å]")
    axes[1].set_ylim(-5, 5)
    plt.tight_layout()
    plt.show()

In [ ]:
# plot corner plot of continuum fit parameters
# Only plot the physically interesting parameters (not logsfr_ratios)
# param_names = ['logzsol', 'dust2', 'logmass', 'dust_ratio', 'dust_index', 'sigma_smooth']
# param_idx   = [model.theta_index[p].start for p in param_names]

# samples_plot = flat_samples_cont[:, param_idx]

# fig = corner.corner(
#     samples_plot,
#     labels=param_names,
#     quantiles=[0.16, 0.5, 0.84],
#     show_titles=True,
#     title_fmt=".3f",
#     smooth=1.0,
# )
# # plt.savefig("corner_plot.png", dpi=150)
# plt.show()

In [ ]:
full_model.theta_labels()

In [ ]:
# Key physical parameters only — skip logsfr_ratios for clarity
param_names = [
    "logzsol",
    "dust2",
    "logmass",
    "dust_ratio",
    "dust_index",
    "sigma_smooth",
    "gas_logz",
    "gas_logu",
    "eline_sigma",
    # PSD params
    "tau_dyn",
    "tau_eq",
    "sigma_dyn",
    "sigma_reg",
    "gas_lognH",
    "gas_logco",
    "gas_logno",
    "gas_logqion",
]
# param_names = full_model.theta_labels()

param_idx = [full_model.theta_index[p].start for p in param_names]
samples_plot = flat_samples_full[:, param_idx]
# samples_plot = flat_samples_full

fig = corner.corner(
    samples_plot,
    labels=param_names,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt=".3f",
    smooth=1.0,
    color="C0",
)
# plt.savefig("corner_full_nebular.png", dpi=150)
plt.show()

In [ ]:
# import emcee
# from prospect.fitting import lnprobfn
# import warnings
# import numpy as np

# # unflatten samples for continuing emcee
# chain = flat_samples_full.reshape(-1, 64, flat_samples_full.shape[1])
# print(f"Unflattened chain shape: {chain.shape}")

# # Continue sampling with emcee
# nwalkers, ndim = chain.shape[1], chain.shape[2]
# print(f"Continuing sampling with emcee: {nwalkers} walkers, {ndim} dimensions")

# # Last walker positions
# p0 = chain[-1, :, :]  # shape: (nwalkers, ndim)
# print(f"Starting positions shape: {p0.shape}")

# # Rebuild the likelihood function
# def lnp_full(theta):
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore", RuntimeWarning)
#         try:
#             lp = lnprobfn(theta, model=full_model, obs=obs,
#                           sps=sps, nested=False)
#             return lp if np.isfinite(lp) else -np.inf
#         except Exception:
#             return -np.inf

# # Continue sampling — no burn-in needed since walkers are already warm
# sampler_continue = emcee.EnsembleSampler(nwalkers, ndim, lnp_full)
# n_more_steps = 10000
# sampler_continue.run_mcmc(p0, n_more_steps, progress=True,
#                            skip_initial_state_check=True)

# # Combine old + new
# new_flat = sampler_continue.get_chain(flat=True, discard=1000, thin=5)

In [ ]:
# # corner
# param_names = ['logzsol', 'dust2', 'logmass', 'dust_ratio',
#                'dust_index', 'sigma_smooth', 'gas_logz', 'gas_logu',
#                'eline_sigma',
#                #PSD params
#                'tau_dyn', 'tau_eq', 'sigma_dyn', 'sigma_reg'
#                ]
# param_idx = [full_model.theta_index[p].start for p in param_names]
# samples_plot = new_flat[:, param_idx]
# fig = corner.corner(
#     samples_plot,
#     labels=param_names,
#     quantiles=[0.16, 0.5, 0.84],
#     show_titles=True,
#     title_fmt=".3f",
#     smooth=1.0,
#     color='C1',
# )
# # plt.savefig("corner_full_nebular_continued.png", dpi=150)
# plt.show()

In [ ]:
# get 2000 samples from the full posterior
# flat_samples_full = new_flat
n_samples = len(flat_samples_full) if len(flat_samples_full) < 5000 else 5000
# posterior_samples = flat_samples_full
posterior_samples = flat_samples_full[
    np.random.choice(len(flat_samples_full), size=n_samples, replace=False)
]

sps = P.build_cue_sps()

# Generate SED for each sample
spec_samples = []
for i, theta in enumerate(posterior_samples):
    print(f"Generating SED {i}/{n_samples}", end="\r", flush=True)
    full_model.set_parameters(theta)
    pred_spec_i, _ = full_model.predict(theta, observations=obs_full, sps=sps)
    spec_samples.append(pred_spec_i[0])

In [ ]:
# # save new flat samples and spec_samples
# import pickle
# with open("../../results/39632946302291248_wider_init/continued_samples.pkl", "wb") as f:
#     pickle.dump({
#         "flat_samples_full": flat_samples_full,
#         "spec_samples": spec_samples,
#     }, f)


In [ ]:
from hubersed.conversion import maggies_to_flambda

# Percentiles for plotting
data = spec_maggies

# import pickle
# with open("../../results/39632946302291248_wider_init/continued_samples.pkl", "rb") as f:
#     saved = pickle.load(f)
#     flat_samples_full = saved["flat_samples_full"]
#     spec_samples = saved["spec_samples"]

spec_median = np.median(spec_samples, axis=0)
spec_lo = np.percentile(spec_samples, 16, axis=0)
spec_hi = np.percentile(spec_samples, 84, axis=0)

# Plot
fig, (ax, ax2) = plt.subplots(
    2, 1, figsize=(12, 4), constrained_layout=True, sharex=True
)
ax.step(wave, spec, where="mid", c="k", lw=0.5, alpha=0.5, label="Data")
# ax.fill_between(wave, spec_lo, spec_hi, color='C1', alpha=1, label='16–84% posterior')
ax.step(
    wave,
    maggies_to_flambda(wave, spec_median) / 1e-17,
    where="mid",
    c="C0",
    lw=1,
    label="Median",
)
ax.axvline(
    8350.1,
    c="red",
    lw=0.8,
    ls="--",
)
ax.axvline(
    8305,
    c="orange",
    lw=0.8,
    ls="--",
)
ax.set_ylabel(r"Flux [$10^{-17}$ erg/s/cm$^2$/\AA]")
ax.legend(frameon=False)

# ivar
ax2.step(
    wave,
    ivar_maggies,
    where="mid",
    lw=0.5,
    c="grey",
    alpha=0.7,
    label="Inverse Variance",
)
# vertical line at 8350.1, 8305
ax2.axvline(8350.1, c="red", lw=0.8, ls="--", label="Sky line 8350.1 Å")
ax2.axvline(8305, c="orange", lw=0.8, ls="--", label="Sky line 8305 Å")
# ylabel in maggies
ax2.set_ylabel("Inverse Variance [1/Maggies2]")
ax2.set_xlabel("Wavelength [Å]")
plt.tight_layout()
plt.show()

In [ ]:
# convert to rest-frame wavelength
wave_rest = wave / (1 + redshift)
data_rest = (
    maggies_to_flambda(wave, spec_maggies) / (1 + redshift) / 1e-17
)  # spec_maggies is in observed frame

spec_median_rest = maggies_to_flambda(wave, spec_median) / (1 + redshift) / 1e-17
spec_lo_rest = maggies_to_flambda(wave, spec_lo) / (1 + redshift) / 1e-17
spec_hi_rest = maggies_to_flambda(wave, spec_hi) / (1 + redshift) / 1e-17
# spec_median_rest_LSF = R_MATRIX.dot(spec_median_rest)

diagnostic_lines = {
    r"H$\alpha$ $\lambda$6563": 6564.6,
    r"[NII] $\lambda$6584": 6585.3,
    r"[OIII] $\lambda$5007": 5008.2,
    r"H$\beta$ $\lambda$4861": 4862.7,
    r"[SII] $\lambda$6717": 6718.3,
    r"[SII] $\lambda$6731": 6732.7,
    r"[OII] $\lambda$3726": 3727.1,
    r"[OII] $\lambda$3729": 3730.1,
}
fig, ax = plt.subplots(
    nrows=2,
    ncols=4,
    figsize=(10, 4),
    dpi=300,
    constrained_layout=True,
)

for i, (line_name, line_wave) in enumerate(diagnostic_lines.items()):
    row, col = divmod(i, 4)
    a = ax[row, col]
    mask_line = (wave_rest > line_wave - 15) & (wave_rest < line_wave + 15) & mask

    a.step(
        wave_rest[mask_line],
        data_rest[mask_line],
        where="mid",
        c="k",
        lw=1.5,
        label="Data",
    )
    a.fill_between(
        wave_rest[mask_line],
        spec_lo_rest[mask_line],
        spec_hi_rest[mask_line],
        color="C1",
        alpha=0.3,
        label="16--84",
    )
    a.step(
        wave_rest[mask_line],
        spec_median_rest[mask_line],
        where="mid",
        c="C0",
        lw=1,
        label="Median",
    )

    # With LSF
    # a.step(wave_rest[mask_line], spec_median_rest_LSF[mask_line],
    #        where='mid', c='C0', lw=1.5, label='Median', alpha=0.9)

    a.set_title(line_name, fontsize=10)

    # set x lim
    a.set_xlim(line_wave - 15, line_wave + 15)

    # only label edges
    if row == 1:
        a.set_xlabel(r"Rest-frame Wavelength [\AA]")
    else:
        a.tick_params(labelbottom=True)

    if col == 0:
        a.set_ylabel(r"Flux [$10^{-17}$ erg/s/cm$^2$/\AA]")
    else:
        a.tick_params(labelleft=False)

# single legend at the top
handles, labels = ax[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=4,
    frameon=False,
    fontsize=12,
    bbox_to_anchor=(0.5, 1.07),
)